# DecodeMultiResizeCropEmbed: `coords` for Explicit Placement

The `coords` parameter lets you specify **exact pixel center positions** for
where a crop is placed on the canvas, instead of sampling from a range.

- `coords=[(112, 224)]` -- deterministic: center crop at pixel (112, 224)
- `coords=[(0, 160), (160, 160), (320, 160)]` -- random choice: each image picks one coord

This is useful for eval pipelines where you want reproducible, interpretable
crop placement (e.g., "always left-aligned" or "always centered vertically").

Overflow (crop extends beyond canvas edge) is handled by clipping the source crop.

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeMultiResizeCropEmbed,
    MultiCropPipeline,
    ToTorchImage,
    RandomBackgroundBlend,
    IMAGENET_MEAN,
    IMAGENET_STD,
    show_batch,
    show_rgba,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

## 1. Single Coord: Deterministic Placement

Place the same 224px crop at two fixed positions on a 448px canvas:
- `left` centered at pixel (112, 224)
- `right` centered at pixel (336, 224)

Both views use the same crop seed so the crop content is identical --
only the canvas placement differs.

In [ ]:
views = {
    'left_224': dict(size=224, x_range=0.5, y_range=0.5, coords=[(112, 224)], seed=42, embed_seed=100),
    'right_224': dict(size=224, x_range=0.5, y_range=0.5, coords=[(336, 224)], seed=42, embed_seed=101),
}

decoder = DecodeMultiResizeCropEmbed(views, canvas_size=448)
print(decoder)

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=True,
    pipelines={'image': [decoder]},
    exclude_fields=['path'], verbose=False,
)
batch = next(iter(loader))
loader.shutdown()

# Verify placement from embed_rects
params = decoder.last_params
for name in views:
    rects = params['embed_rects'][name]
    print(f"{name}: x0={rects[0, 0]}, y0={rects[0, 1]} (all images same: {np.all(rects == rects[0])})")

show_rgba(
    [batch['left_224'], batch['right_224']],
    row_labels=['left (112, 224)', 'right (336, 224)'],
    suptitle='Single coord: deterministic left vs right placement',
)

## 2. Mixed Sizes with Coords

Different crop sizes placed at specific positions. The smaller crop has
more slack on the canvas, so the same fractional coord maps to a different
pixel offset.

In [ ]:
views = {
    'large_left': dict(size=224, coords=[(112, 224)], seed=42, embed_seed=100),
    'large_right': dict(size=224, coords=[(336, 224)], seed=42, embed_seed=101),
    'small_left': dict(size=112, coords=[(112, 224)], seed=43, embed_seed=102),
}

decoder = DecodeMultiResizeCropEmbed(views, canvas_size=448)

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=True,
    pipelines={'image': [decoder]},
    exclude_fields=['path'], verbose=False,
)
batch = next(iter(loader))
loader.shutdown()

params = decoder.last_params
for name in views:
    r = params['embed_rects'][name][0]
    print(f"{name}: x0={r[0]}, y0={r[1]}, w={r[2]}, h={r[3]}")

show_rgba(
    [batch['large_left'], batch['large_right'], batch['small_left']],
    row_labels=['large left (224)', 'large right (224)', 'small left (112)'],
    suptitle='Mixed sizes: coords place each crop independently',
)

## 3. Multiple Coords: Random Choice per Image

With multiple coords, each image in the batch randomly picks one.
This gives controlled stochasticity -- the crop can only appear at
one of the specified pixel positions.

In [ ]:
views = {
    'view': dict(
        size=160,
        coords=[(0, 160), (160, 160), (320, 160)],  # left, center, or right
        seed=42,
        embed_seed=100,
    ),
}

decoder = DecodeMultiResizeCropEmbed(views, canvas_size=320)

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=True,
    pipelines={'image': [decoder]},
    exclude_fields=['path'], verbose=False,
)
batch = next(iter(loader))
loader.shutdown()

# Show which position each image got
params = decoder.last_params
rects = params['embed_rects']['view']
for i in range(8):
    x0 = rects[i, 0]
    label = {0: 'left', 80: 'center', 240: 'right'}.get(x0, f'x0={x0}')
    print(f"  image {i}: x0={x0} ({label})")

show_rgba(
    [batch['view']],
    row_labels=['3 coords: left/center/right'],
    suptitle='Multiple coords: each image randomly picks a position',
)

## 4. Full Pipeline: Coords + Blend + Normalize

Complete pipeline matching the user's original use case: coords-based
placement through `DecodeMultiResizeCropEmbed` into `RandomBackgroundBlend`.

In [ ]:
views = {
    'view1_224_left': dict(size=224, x_range=.5, y_range=.5, coords=[(112, 224)], seed=42, embed_seed=100, size_mode='per_image'),
    'view1_224_right': dict(size=224, x_range=.5, y_range=.5, coords=[(336, 224)], seed=43, embed_seed=101, size_mode='per_image'),
    'view1_112_left': dict(size=112, x_range=.5, y_range=.5, coords=[(112, 224)], seed=42, embed_seed=100, size_mode='per_image'),
}
view_names = list(views.keys())

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=True,
    pipelines={'image': [
        DecodeMultiResizeCropEmbed(
            views,
            canvas_size=448,
        ),
        MultiCropPipeline({
            'view1_224_left': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(background='zeros'),
            ],
            'view1_112_left': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(background='zeros'),
            ],
            'view1_224_right': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(background='zeros'),
            ],
        }),
    ]},
    exclude_fields=['path'],
    verbose=False,
)
batch = next(iter(loader))
loader.shutdown()

for name in view_names:
    print(f"{name}: {batch[name].shape}, dtype={batch[name].dtype}")

show_batch(
    batch, view_names,
    suptitle='Full pipeline: coords placement + zeros background',
)

## 5. Coords vs Range: Side-by-Side

Compare `coords=[(80, 160)]` (deterministic) with `embed_x_range=(0, 1)`
(random uniform). Coords give the same position every time; ranges vary.

In [ ]:
# Deterministic coords
dec_coords = DecodeMultiResizeCropEmbed(
    {'view': dict(size=160, coords=[(80, 160)], seed=42, embed_seed=100)},
    canvas_size=320,
)

# Random range
dec_range = DecodeMultiResizeCropEmbed(
    {'view': dict(size=160, seed=42, embed_seed=100)},
    canvas_size=320,
    embed_x_range=(0, 1), embed_y_range=(0, 1),
)

rows = []
labels = []
for dec, label in [(dec_coords, 'coords=[(80, 160)]'), (dec_range, 'embed_x/y_range=(0, 1)')]:
    loader = SlipstreamLoader(
        dataset, batch_size=8, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    b = next(iter(loader))
    rows.append(b['view'])
    labels.append(label)
    loader.shutdown()

show_rgba(rows, row_labels=labels,
          suptitle='Coords (deterministic) vs range (random)')

## 6. Grid of Positions

Sweep over a grid of pixel center positions to visualize the full placement space.
Canvas is 320px, crop is 160px.

In [ ]:
x_positions = [25, 75, 125, 175, 225]
y_positions = [125, 175, 225]
rows = []
labels = []

for cy in y_positions:
    for cx in x_positions:
        dec = DecodeMultiResizeCropEmbed(
            {'view': dict(size=50, coords=[(cx, cy)], seed=42, embed_seed=100)},
            canvas_size=320,
        )
        loader = SlipstreamLoader(
            dataset, batch_size=1, shuffle=False,
            pipelines={'image': [dec]},
            exclude_fields=['path'], verbose=False,
        )
        b = next(iter(loader))
        rows.append(b['view'])
        labels.append(f'({cx}, {cy})')
        loader.shutdown()

# Display as 3x5 grid
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for idx, (arr, label) in enumerate(zip(rows, labels)):
    r, c = divmod(idx, 5)
    img = arr[0]
    alpha = img[:, :, 3:4].astype(np.float32) / 255.0
    rgb = img[:, :, :3].astype(np.float32)
    bg = np.full_like(rgb, 200)
    composite = (alpha * rgb + (1 - alpha) * bg).astype(np.uint8)
    axes[r, c].imshow(composite)
    axes[r, c].set_title(label, fontsize=10)
    axes[r, c].axis('off')
fig.suptitle('Grid of coords: (cx, cy) pixel center positions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()